In [9]:
import os
import pandas as pd
from dotenv import load_dotenv
from google import genai

In [8]:
load_dotenv(".env")

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env")

client = genai.Client(api_key=API_KEY)


def load_csv(file_path):
    try:
        data = pd.read_csv(file_path)
        print(f"CSV loaded successfully: {data.shape[0]} rows, {data.shape[1]} columns.")
        return data
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return None


def profile_data(data):
    lines = []
    lines.append(f"Shape: {data.shape[0]} rows, {data.shape[1]} columns")
    lines.append(f"Column types:\n{data.dtypes.to_string()}")

    missing = data.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        lines.append(f"Missing values per column:\n{missing.to_string()}")
    else:
        lines.append("No missing values in any column.")

    numeric_cols = data.select_dtypes(include="number").columns
    if len(numeric_cols) > 0:
        lines.append(f"Numeric summary:\n{data[numeric_cols].describe().to_string()}")

    if len(numeric_cols) > 1:
        lines.append(f"Correlation matrix:\n{data[numeric_cols].corr().round(2).to_string()}")

    categorical_cols = data.select_dtypes(include="object").columns
    for col in categorical_cols[:5]:  # cap so the profile doesn't explode in size
        top = data[col].value_counts().head(5)
        lines.append(f"Top values in '{col}':\n{top.to_string()}")

    return "\n\n".join(lines)


def try_direct_computation(data, question):
    q = question.lower()
    mentioned_cols = [col for col in data.columns if col.lower() in q]
    if not mentioned_cols:
        return None

    col = mentioned_cols[0]
    is_numeric = pd.api.types.is_numeric_dtype(data[col])

    if is_numeric:
        if any(w in q for w in ["average", "mean"]):
            return f"The average {col} is {data[col].mean():.2f}."
        if any(w in q for w in ["sum", "total"]) and "by" not in q and "per" not in q:
            return f"The total {col} is {data[col].sum():.2f}."
        if any(w in q for w in ["max", "highest", "maximum"]):
            return f"The maximum {col} is {data[col].max()}."
        if any(w in q for w in ["min", "lowest", "minimum"]):
            return f"The minimum {col} is {data[col].min()}."
        if "median" in q:
            return f"The median {col} is {data[col].median():.2f}."
        if "standard deviation" in q or "std" in q:
            return f"The standard deviation of {col} is {data[col].std():.2f}."

    if "unique" in q:
        return f"There are {data[col].nunique()} unique values in {col}."
    if "how many" in q or "count" in q:
        return f"There are {data[col].count()} non-missing values in {col}."

    if "top" in q or "most common" in q or "most frequent" in q:
        top_values = data[col].value_counts().head(5)
        lines = [f"  {idx}: {val}" for idx, val in top_values.items()]
        return f"Top values in {col}:\n" + "\n".join(lines)

    
    if (" by " in q or " per " in q) and len(mentioned_cols) >= 2:
        agg_col, group_col = mentioned_cols[0], mentioned_cols[1]
        if any(w in q for w in ["average", "mean"]):
            result = data.groupby(group_col)[agg_col].mean().sort_values(ascending=False)
        elif any(w in q for w in ["sum", "total"]):
            result = data.groupby(group_col)[agg_col].sum().sort_values(ascending=False)
        else:
            result = data.groupby(group_col)[agg_col].count().sort_values(ascending=False)
        lines = [f"  {idx}: {val:.2f}" for idx, val in result.head(10).items()]
        return f"{agg_col} grouped by {group_col} (top 10):\n" + "\n".join(lines)

    return None


def ask_ai(data, question, profile):
    columns = ", ".join(data.columns.tolist())

    prompt = f"""
You are a CSV analyst.

Dataset columns:
{columns}

Computed statistics:
{profile}

User question:
{question}

Use the column names and computed statistics to answer.
If the information is insufficient, say so instead of guessing.
"""

    try:
        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f"Error generating response: {e}"


def ask_csv_question(data, question, profile):
    direct_answer = try_direct_computation(data, question)
    if direct_answer:
        print("\n[Computed directly from the data]")
        print(direct_answer)
    else:
        print("\n[AI-interpreted from computed statistics]")
        print(ask_ai(data, question, profile))


def main():
    csv_file = input("Enter the path to your CSV file: ").strip().strip('"')
    data = load_csv(csv_file)
    if data is None:
        return

    print("\nComputing dataset profile...")
    profile = profile_data(data)
    print("Profile ready.\n")

    print("You can now ask questions about the CSV data.")
    print("Try: 'average of <column>', 'top values in <column>', 'total <column> by <other column>',")
    print("or open-ended questions like 'what trends do you see?'\n")

    while True:
        question = input("Ask a question (or type 'exit' to quit): ")
        if question.lower() == "exit":
            print("Goodbye!")
            break
        ask_csv_question(data, question, profile)


if __name__ == "__main__":
    main()

Enter the path to your CSV file:  C:\Users\prade\Downloads\Drug_overdose_death_rates__by_drug_type__sex__age__race__and_Hispanic_origin__United_States (2).csv


CSV loaded successfully: 6228 rows, 15 columns.

Computing dataset profile...
Profile ready.

You can now ask questions about the CSV data.
Try: 'average of <column>', 'top values in <column>', 'total <column> by <other column>',
or open-ended questions like 'what trends do you see?'



Ask a question (or type 'exit' to quit):  what kind of file is this?



[AI-interpreted from computed statistics]
Based on the provided statistics, there is not enough information to determine the exact **file format** (such as CSV, Excel, JSON, or Parquet). 

However, in terms of content, it is a tabular dataset containing public health statistics on **drug overdose death rates** (covering various drug panels, demographic groupings/stubs, unit types, and years from 1999 to 2018).


Ask a question (or type 'exit' to quit):  i am just asking about the data not file type



[AI-interpreted from computed statistics]
Based on the provided statistics, here is an overview of the dataset:

### **Domain & Content**
* **Topic:** Drug overdose death rates in the United States (all 6,228 rows belong to the indicator *"Drug overdose death rates"*).
* **Specific Panels/Subtypes:** Covers categories such as:
  * All drug overdose deaths
  * Drug overdose deaths involving any opioid
  * Drug overdose deaths involving natural and semisynthetic opioids
  * Drug overdose deaths involving methadone
  * Drug overdose deaths involving other synthetic opioids (other than methadone)

### **Key Metrics & Statistics (`ESTIMATE`)**
* **Measurement Units:** 
  * *Deaths per 100,000 resident population, crude* (3,600 rows)
  * *Deaths per 100,000 resident population, age-adjusted* (2,628 rows)
* **Value Distribution:**
  * **Mean:** 4.74 deaths per 100,000
  * **Median (50%):** 2.10 deaths per 100,000
  * **Range:** 0.0 to 54.3 deaths per 100,000
  * **Missing Values:** 1,111 mis

Ask a question (or type 'exit' to quit):  exit


Goodbye!
